# Clase 17 — DataFrames: Operaciones y Agregaciones

Notebook con **todos los ejercicios y casos de uso** propuestos en las dos sesiones de la Clase 17:

- **Sesión 1** — DataFrames: operaciones básicas, pipelines, caso *MobilityData Analytics*
- **Sesión 2** — Agregaciones, `groupBy`, `agg`, ventanas, `rollup`, `cube`, `pivot`, caso *MarketNova Analytics*

Entorno: Apache Spark 4.1.1 + Scala 2.13 (kernel Almond) en modo `local[*]`.


---

# 🟢 Sesión 1 — DataFrames: Operaciones básicas

## 0. Inicialización de Spark


In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Clase17-Sesion1-DataFrames")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

println(s"✅ Spark ${spark.version} listo")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 22:47:33 INFO SparkContext: Running Spark version 4.1.1
26/05/03 22:47:33 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/03 22:47:33 INFO SparkContext: Java version 17.0.18+8
26/05/03 22:47:33 INFO ResourceUtils: ==============================================================
26/05/03 22:47:33 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/03 22:47:33 INFO ResourceUtils: ==============================================================
26/05/03 22:47:33 INFO SparkContext: Submitted application: Clase17-Sesion1-DataFrames
26/05/03 22:47:33 INFO SecurityManager: Changing view acls to: gre
26/05/03 22:47:33 INFO SecurityManager: Changing modify acls to: gre
26/05/03 22:47:33 INFO SecurityManager: Changing view acls groups to: gre
26/05/03 22:47:33 INFO SecurityManager: Changing modify acls groups to: gre
26/05/03 22:47:33 INFO SecurityManager: SecurityManager: authen

✅ Spark 4.1.1 listo


import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@4f2351d4
import spark.implicits._

## 📂 Dataset de práctica — `dfVentas`

In [2]:
val dfVentas = Seq(
  (1,"Laptop","Tecnología",1200.0,2,"Madrid"),
  (2,"Teclado","Tecnología",45.0,5,"Valencia"),
  (3,"Monitor","Tecnología",350.0,1,"Madrid"),
  (4,"Silla","Oficina",120.0,4,"Barcelona"),
  (5,"Mesa","Oficina",250.0,2,"Sevilla"),
  (6,"Laptop","Tecnología",1200.0,1,"Madrid"),
  (7,"Tablet","Tecnología",600.0,2,"Bilbao"),
  (8,"Ratón","Tecnología",25.0,10,"Madrid"),
  (9,"Impresora","Tecnología",200.0,1,"Sevilla"),
  (10,"Silla","Oficina",120.0,3,"Madrid"),
  (11,"Mesa","Oficina",250.0,1,"Barcelona"),
  (12,"Laptop","Tecnología",1200.0,1,"Valencia"),
  (13,"Monitor","Tecnología",350.0,2,"Madrid"),
  (14,"Tablet","Tecnología",600.0,1,"Sevilla"),
  (15,"Ratón","Tecnología",25.0,8,"Bilbao"),
  (16,"Teclado","Tecnología",45.0,6,"Madrid"),
  (17,"Silla","Oficina",120.0,2,"Valencia"),
  (18,"Mesa","Oficina",250.0,3,"Madrid"),
  (19,"Laptop","Tecnología",1200.0,1,"Barcelona"),
  (20,"Monitor","Tecnología",350.0,1,"Sevilla"),
  (21,"Tablet","Tecnología",600.0,2,"Madrid"),
  (22,"Ratón","Tecnología",25.0,12,"Madrid"),
  (23,"Teclado","Tecnología",45.0,4,"Bilbao"),
  (24,"Silla","Oficina",120.0,1,"Sevilla"),
  (25,"Mesa","Oficina",250.0,2,"Madrid"),
  (26,"Laptop","Tecnología",1200.0,3,"Madrid"),
  (27,"Monitor","Tecnología",350.0,2,"Barcelona"),
  (28,"Tablet","Tecnología",600.0,1,"Valencia"),
  (29,"Ratón","Tecnología",25.0,15,"Madrid"),
  (30,"Teclado","Tecnología",45.0,7,"Sevilla"),
  (31,"Silla","Oficina",120.0,5,"Madrid"),
  (32,"Mesa","Oficina",250.0,1,"Bilbao"),
  (33,"Laptop","Tecnología",1200.0,2,"Sevilla"),
  (34,"Monitor","Tecnología",350.0,3,"Madrid"),
  (35,"Tablet","Tecnología",600.0,2,"Barcelona"),
  (36,"Ratón","Tecnología",25.0,9,"Valencia"),
  (37,"Teclado","Tecnología",45.0,6,"Madrid"),
  (38,"Silla","Oficina",120.0,2,"Bilbao"),
  (39,"Mesa","Oficina",250.0,2,"Madrid"),
  (40,"Laptop","Tecnología",1200.0,1,"Madrid")
).toDF("id","producto","categoria","precio","cantidad","ciudad")

println(s"Filas: ${dfVentas.count()}")

Filas: 40


dfVentas: org.apache.spark.sql.package.DataFrame = [id: int, producto: string ... 4 more fields]

## 🧪 BLOQUE 1 — Exploración

In [3]:
dfVentas.show(5)

+---+--------+----------+------+--------+---------+
| id|producto| categoria|precio|cantidad|   ciudad|
+---+--------+----------+------+--------+---------+
|  1|  Laptop|Tecnología|1200.0|       2|   Madrid|
|  2| Teclado|Tecnología|  45.0|       5| Valencia|
|  3| Monitor|Tecnología| 350.0|       1|   Madrid|
|  4|   Silla|   Oficina| 120.0|       4|Barcelona|
|  5|    Mesa|   Oficina| 250.0|       2|  Sevilla|
+---+--------+----------+------+--------+---------+
only showing top 5 rows


In [4]:
dfVentas.printSchema()

root
 |-- id: integer (nullable = false)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- precio: double (nullable = false)
 |-- cantidad: integer (nullable = false)
 |-- ciudad: string (nullable = true)



In [5]:
dfVentas.describe("precio", "cantidad").show()

+-------+-----------------+------------------+
|summary|           precio|          cantidad|
+-------+-----------------+------------------+
|  count|               40|                40|
|   mean|            398.0|             3.475|
| stddev|413.5831544454219|3.3203915431767985|
|    min|             25.0|                 1|
|    max|           1200.0|                15|
+-------+-----------------+------------------+



In [6]:
dfVentas.columns.foreach(println)

id
producto
categoria
precio
cantidad
ciudad


## 🧪 BLOQUE 2 — Operaciones básicas

1. **Selección** • 2. **Filtrado** • 3. **Nueva columna** • 4. **Ordenación**


In [7]:
// 1. Selección
dfVentas.select("producto", "precio").show()

+---------+------+
| producto|precio|
+---------+------+
|   Laptop|1200.0|
|  Teclado|  45.0|
|  Monitor| 350.0|
|    Silla| 120.0|
|     Mesa| 250.0|
|   Laptop|1200.0|
|   Tablet| 600.0|
|    Ratón|  25.0|
|Impresora| 200.0|
|    Silla| 120.0|
|     Mesa| 250.0|
|   Laptop|1200.0|
|  Monitor| 350.0|
|   Tablet| 600.0|
|    Ratón|  25.0|
|  Teclado|  45.0|
|    Silla| 120.0|
|     Mesa| 250.0|
|   Laptop|1200.0|
|  Monitor| 350.0|
+---------+------+
only showing top 20 rows


In [8]:
// 2. Filtrado
dfVentas.filter($"precio" > 500).show()

+---+--------+----------+------+--------+---------+
| id|producto| categoria|precio|cantidad|   ciudad|
+---+--------+----------+------+--------+---------+
|  1|  Laptop|Tecnología|1200.0|       2|   Madrid|
|  6|  Laptop|Tecnología|1200.0|       1|   Madrid|
|  7|  Tablet|Tecnología| 600.0|       2|   Bilbao|
| 12|  Laptop|Tecnología|1200.0|       1| Valencia|
| 14|  Tablet|Tecnología| 600.0|       1|  Sevilla|
| 19|  Laptop|Tecnología|1200.0|       1|Barcelona|
| 21|  Tablet|Tecnología| 600.0|       2|   Madrid|
| 26|  Laptop|Tecnología|1200.0|       3|   Madrid|
| 28|  Tablet|Tecnología| 600.0|       1| Valencia|
| 33|  Laptop|Tecnología|1200.0|       2|  Sevilla|
| 35|  Tablet|Tecnología| 600.0|       2|Barcelona|
| 40|  Laptop|Tecnología|1200.0|       1|   Madrid|
+---+--------+----------+------+--------+---------+



In [9]:
// 3. Nueva columna `total`
val dfTotal = dfVentas.withColumn("total", $"precio" * $"cantidad")
dfTotal.show(5)

+---+--------+----------+------+--------+---------+------+
| id|producto| categoria|precio|cantidad|   ciudad| total|
+---+--------+----------+------+--------+---------+------+
|  1|  Laptop|Tecnología|1200.0|       2|   Madrid|2400.0|
|  2| Teclado|Tecnología|  45.0|       5| Valencia| 225.0|
|  3| Monitor|Tecnología| 350.0|       1|   Madrid| 350.0|
|  4|   Silla|   Oficina| 120.0|       4|Barcelona| 480.0|
|  5|    Mesa|   Oficina| 250.0|       2|  Sevilla| 500.0|
+---+--------+----------+------+--------+---------+------+
only showing top 5 rows


dfTotal: org.apache.spark.sql.package.DataFrame = [id: int, producto: string ... 5 more fields]

In [10]:
// 4. Ordenación
dfTotal.orderBy($"total".desc).show()

+---+--------+----------+------+--------+---------+------+
| id|producto| categoria|precio|cantidad|   ciudad| total|
+---+--------+----------+------+--------+---------+------+
| 26|  Laptop|Tecnología|1200.0|       3|   Madrid|3600.0|
| 33|  Laptop|Tecnología|1200.0|       2|  Sevilla|2400.0|
|  1|  Laptop|Tecnología|1200.0|       2|   Madrid|2400.0|
| 40|  Laptop|Tecnología|1200.0|       1|   Madrid|1200.0|
|  6|  Laptop|Tecnología|1200.0|       1|   Madrid|1200.0|
| 21|  Tablet|Tecnología| 600.0|       2|   Madrid|1200.0|
|  7|  Tablet|Tecnología| 600.0|       2|   Bilbao|1200.0|
| 12|  Laptop|Tecnología|1200.0|       1| Valencia|1200.0|
| 19|  Laptop|Tecnología|1200.0|       1|Barcelona|1200.0|
| 35|  Tablet|Tecnología| 600.0|       2|Barcelona|1200.0|
| 34| Monitor|Tecnología| 350.0|       3|   Madrid|1050.0|
| 18|    Mesa|   Oficina| 250.0|       3|   Madrid| 750.0|
| 13| Monitor|Tecnología| 350.0|       2|   Madrid| 700.0|
| 27| Monitor|Tecnología| 350.0|       2|Barcelona| 700.

## 🧪 Ejercicios basados en preguntas de negocio

### 🔹 Ejercicio 1 — Mostrar solo ventas de Madrid

In [11]:
dfVentas.filter($"ciudad" === "Madrid").show()

+---+--------+----------+------+--------+------+
| id|producto| categoria|precio|cantidad|ciudad|
+---+--------+----------+------+--------+------+
|  1|  Laptop|Tecnología|1200.0|       2|Madrid|
|  3| Monitor|Tecnología| 350.0|       1|Madrid|
|  6|  Laptop|Tecnología|1200.0|       1|Madrid|
|  8|   Ratón|Tecnología|  25.0|      10|Madrid|
| 10|   Silla|   Oficina| 120.0|       3|Madrid|
| 13| Monitor|Tecnología| 350.0|       2|Madrid|
| 16| Teclado|Tecnología|  45.0|       6|Madrid|
| 18|    Mesa|   Oficina| 250.0|       3|Madrid|
| 21|  Tablet|Tecnología| 600.0|       2|Madrid|
| 22|   Ratón|Tecnología|  25.0|      12|Madrid|
| 25|    Mesa|   Oficina| 250.0|       2|Madrid|
| 26|  Laptop|Tecnología|1200.0|       3|Madrid|
| 29|   Ratón|Tecnología|  25.0|      15|Madrid|
| 31|   Silla|   Oficina| 120.0|       5|Madrid|
| 34| Monitor|Tecnología| 350.0|       3|Madrid|
| 37| Teclado|Tecnología|  45.0|       6|Madrid|
| 39|    Mesa|   Oficina| 250.0|       2|Madrid|
| 40|  Laptop|Tecnol

### 🔹 Ejercicio 2 — Crear columna IVA (21%)

In [ ]:
dfVentas.withColumn("precio_iva", $"precio" * 1.21).show()

### 🔹 Ejercicio 3 — Eliminar columna `categoria`

In [12]:
dfVentas.drop("categoria").show()

+---+---------+------+--------+---------+
| id| producto|precio|cantidad|   ciudad|
+---+---------+------+--------+---------+
|  1|   Laptop|1200.0|       2|   Madrid|
|  2|  Teclado|  45.0|       5| Valencia|
|  3|  Monitor| 350.0|       1|   Madrid|
|  4|    Silla| 120.0|       4|Barcelona|
|  5|     Mesa| 250.0|       2|  Sevilla|
|  6|   Laptop|1200.0|       1|   Madrid|
|  7|   Tablet| 600.0|       2|   Bilbao|
|  8|    Ratón|  25.0|      10|   Madrid|
|  9|Impresora| 200.0|       1|  Sevilla|
| 10|    Silla| 120.0|       3|   Madrid|
| 11|     Mesa| 250.0|       1|Barcelona|
| 12|   Laptop|1200.0|       1| Valencia|
| 13|  Monitor| 350.0|       2|   Madrid|
| 14|   Tablet| 600.0|       1|  Sevilla|
| 15|    Ratón|  25.0|       8|   Bilbao|
| 16|  Teclado|  45.0|       6|   Madrid|
| 17|    Silla| 120.0|       2| Valencia|
| 18|     Mesa| 250.0|       3|   Madrid|
| 19|   Laptop|1200.0|       1|Barcelona|
| 20|  Monitor| 350.0|       1|  Sevilla|
+---+---------+------+--------+---

### 🔹 Ejercicio 4 — Renombrar `precio` → `precio_unitario`

In [13]:
dfVentas.withColumnRenamed("precio", "precio_unitario").show()

+---+---------+----------+---------------+--------+---------+
| id| producto| categoria|precio_unitario|cantidad|   ciudad|
+---+---------+----------+---------------+--------+---------+
|  1|   Laptop|Tecnología|         1200.0|       2|   Madrid|
|  2|  Teclado|Tecnología|           45.0|       5| Valencia|
|  3|  Monitor|Tecnología|          350.0|       1|   Madrid|
|  4|    Silla|   Oficina|          120.0|       4|Barcelona|
|  5|     Mesa|   Oficina|          250.0|       2|  Sevilla|
|  6|   Laptop|Tecnología|         1200.0|       1|   Madrid|
|  7|   Tablet|Tecnología|          600.0|       2|   Bilbao|
|  8|    Ratón|Tecnología|           25.0|      10|   Madrid|
|  9|Impresora|Tecnología|          200.0|       1|  Sevilla|
| 10|    Silla|   Oficina|          120.0|       3|   Madrid|
| 11|     Mesa|   Oficina|          250.0|       1|Barcelona|
| 12|   Laptop|Tecnología|         1200.0|       1| Valencia|
| 13|  Monitor|Tecnología|          350.0|       2|   Madrid|
| 14|   

### 🔹 Ejercicio 5 — Quitar duplicados por producto

In [14]:
dfVentas.dropDuplicates("producto").show()

+---+---------+----------+------+--------+---------+
| id| producto| categoria|precio|cantidad|   ciudad|
+---+---------+----------+------+--------+---------+
|  9|Impresora|Tecnología| 200.0|       1|  Sevilla|
|  1|   Laptop|Tecnología|1200.0|       2|   Madrid|
|  5|     Mesa|   Oficina| 250.0|       2|  Sevilla|
|  3|  Monitor|Tecnología| 350.0|       1|   Madrid|
|  8|    Ratón|Tecnología|  25.0|      10|   Madrid|
|  4|    Silla|   Oficina| 120.0|       4|Barcelona|
|  7|   Tablet|Tecnología| 600.0|       2|   Bilbao|
|  2|  Teclado|Tecnología|  45.0|       5| Valencia|
+---+---------+----------+------+--------+---------+



### 🔹 Ejercicio 6 — ¿Cuántas filas hay?

In [15]:
dfVentas.count()

res15: Long = 40L

### 🔹 Ejercicio 7 — ¿Qué columnas existen?

In [16]:
dfVentas.columns

res16: Array[String] = Array(
  "id",
  "producto",
  "categoria",
  "precio",
  "cantidad",
  "ciudad"
)

### 🔹 Ejercicio 8 — Ver tipos

In [17]:
dfVentas.dtypes.foreach(println)

(id,IntegerType)
(producto,StringType)
(categoria,StringType)
(precio,DoubleType)
(cantidad,IntegerType)
(ciudad,StringType)


### 🔹 Ejercicio 9 — Estadísticas de `precio`

In [18]:
dfVentas.describe("precio").show()

+-------+-----------------+
|summary|           precio|
+-------+-----------------+
|  count|               40|
|   mean|            398.0|
| stddev|413.5831544454219|
|    min|             25.0|
|    max|           1200.0|
+-------+-----------------+



## 🧪 PIPELINES

### 🔹 Ejercicio 10 — Ventas de mayor valor económico

1. Filtrar productos con `precio > 100`
2. Crear columna `total = precio * cantidad`
3. Seleccionar `producto`, `total`, `ciudad`
4. Ordenar por `total` descendente


In [19]:
val ventasMayorValor = dfVentas
  .filter($"precio" > 100)
  .withColumn("total", $"precio" * $"cantidad")
  .select("producto", "total", "ciudad")
  .orderBy($"total".desc)

ventasMayorValor.show()

+--------+------+---------+
|producto| total|   ciudad|
+--------+------+---------+
|  Laptop|3600.0|   Madrid|
|  Laptop|2400.0|  Sevilla|
|  Laptop|2400.0|   Madrid|
|  Tablet|1200.0|Barcelona|
|  Laptop|1200.0| Valencia|
|  Laptop|1200.0|   Madrid|
|  Laptop|1200.0|Barcelona|
|  Tablet|1200.0|   Madrid|
|  Tablet|1200.0|   Bilbao|
|  Laptop|1200.0|   Madrid|
| Monitor|1050.0|   Madrid|
|    Mesa| 750.0|   Madrid|
| Monitor| 700.0|   Madrid|
| Monitor| 700.0|Barcelona|
|  Tablet| 600.0|  Sevilla|
|   Silla| 600.0|   Madrid|
|  Tablet| 600.0| Valencia|
|    Mesa| 500.0|  Sevilla|
|    Mesa| 500.0|   Madrid|
|    Mesa| 500.0|   Madrid|
+--------+------+---------+
only showing top 20 rows


ventasMayorValor: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [producto: string, total: double ... 1 more field]

### 🔹 Ejercicio 11 — Productos únicos con precio representativo

1. Quitar duplicados por `producto`
2. Seleccionar `producto`, `precio`
3. Ordenar por `precio` descendente


In [20]:
val productosUnicos = dfVentas
  .dropDuplicates("producto")
  .select("producto", "precio")
  .orderBy($"precio".desc)

productosUnicos.show()

+---------+------+
| producto|precio|
+---------+------+
|   Laptop|1200.0|
|   Tablet| 600.0|
|  Monitor| 350.0|
|     Mesa| 250.0|
|Impresora| 200.0|
|    Silla| 120.0|
|  Teclado|  45.0|
|    Ratón|  25.0|
+---------+------+



productosUnicos: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [producto: string, precio: double]

---

## 🏢 Caso de Estudio — *MobilityData Analytics*

Una empresa de movilidad urbana ha recopilado todos los viajes de **mayo** en distintas ciudades. Necesita:

- Identificar viajes que generan **mayor ingreso** (`distancia × precio`).
- Entender en qué ciudades se concentran esos viajes rentables.
- Producir un dataset limpio y ordenado para análisis posteriores.

### 📥 Dataset de viajes


In [21]:
val viajes = Seq(
  (1,"2026-05-01","Madrid",12.5,10.0),
  (2,"2026-05-01","Barcelona",8.0,7.0),
  (3,"2026-05-02","Madrid",15.0,12.0),
  (4,"2026-05-02","Valencia",6.0,6.0),
  (5,"2026-05-03","Sevilla",20.0,15.0),
  (6,"2026-05-03","Madrid",5.0,5.0),
  (7,"2026-05-04","Bilbao",18.0,13.0),
  (8,"2026-05-04","Madrid",22.0,16.0),
  (9,"2026-05-05","Barcelona",10.0,9.0),
  (10,"2026-05-05","Madrid",14.0,11.0),
  (11,"2026-05-06","Valencia",7.0,6.5),
  (12,"2026-05-06","Madrid",16.0,13.0),
  (13,"2026-05-07","Sevilla",25.0,18.0),
  (14,"2026-05-07","Madrid",9.0,8.0),
  (15,"2026-05-08","Barcelona",11.0,9.5),
  (16,"2026-05-08","Bilbao",13.0,10.0),
  (17,"2026-05-09","Madrid",17.0,14.0),
  (18,"2026-05-09","Valencia",8.0,7.5),
  (19,"2026-05-10","Sevilla",19.0,14.0),
  (20,"2026-05-10","Madrid",21.0,17.0),
  (21,"2026-05-11","Barcelona",9.0,8.0),
  (22,"2026-05-11","Madrid",18.0,15.0),
  (23,"2026-05-12","Valencia",6.5,6.0),
  (24,"2026-05-12","Madrid",20.0,16.0),
  (25,"2026-05-13","Bilbao",14.0,11.0),
  (26,"2026-05-13","Madrid",23.0,18.0),
  (27,"2026-05-14","Sevilla",18.0,13.0),
  (28,"2026-05-14","Madrid",7.0,6.0),
  (29,"2026-05-15","Barcelona",12.0,10.0),
  (30,"2026-05-15","Madrid",19.0,15.0)
).toDF("id","fecha","ciudad","distancia_km","precio")

viajes.show(5)

+---+----------+---------+------------+------+
| id|     fecha|   ciudad|distancia_km|precio|
+---+----------+---------+------------+------+
|  1|2026-05-01|   Madrid|        12.5|  10.0|
|  2|2026-05-01|Barcelona|         8.0|   7.0|
|  3|2026-05-02|   Madrid|        15.0|  12.0|
|  4|2026-05-02| Valencia|         6.0|   6.0|
|  5|2026-05-03|  Sevilla|        20.0|  15.0|
+---+----------+---------+------------+------+
only showing top 5 rows


viajes: org.apache.spark.sql.package.DataFrame = [id: int, fecha: string ... 3 more fields]

### 🛠 Pipeline principal

1. Crear columna `ingreso_viaje = distancia_km * precio`
2. Filtrar viajes con `ingreso_viaje > 150`
3. Seleccionar `ciudad`, `ingreso_viaje`
4. Ordenar por `ingreso_viaje` descendente


In [22]:
val viajesIngresosAltos = viajes
  .withColumn("ingreso_viaje", $"distancia_km" * $"precio")
  .filter($"ingreso_viaje" > 150)
  .select("ciudad", "ingreso_viaje")
  .orderBy($"ingreso_viaje".desc)

viajesIngresosAltos.show()

+-------+-------------+
| ciudad|ingreso_viaje|
+-------+-------------+
|Sevilla|        450.0|
| Madrid|        414.0|
| Madrid|        357.0|
| Madrid|        352.0|
| Madrid|        320.0|
|Sevilla|        300.0|
| Madrid|        285.0|
| Madrid|        270.0|
|Sevilla|        266.0|
| Madrid|        238.0|
| Bilbao|        234.0|
|Sevilla|        234.0|
| Madrid|        208.0|
| Madrid|        180.0|
| Madrid|        154.0|
| Bilbao|        154.0|
+-------+-------------+



viajesIngresosAltos: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, ingreso_viaje: double]

### 💡 Preguntas — Exploración

In [23]:
// 1. ¿Cuántos viajes hay?
println(s"Total viajes: ${viajes.count()}")

// 2. ¿Qué columnas existen?
viajes.columns.foreach(println)

// 3. ¿Qué tipos tienen?
viajes.dtypes.foreach(println)

Total viajes: 30
id
fecha
ciudad
distancia_km
precio
(id,IntegerType)
(fecha,StringType)
(ciudad,StringType)
(distancia_km,DoubleType)
(precio,DoubleType)


### 💡 Preguntas — Transformación

In [24]:
// 1. ¿Cuántos viajes cumplen el filtro (ingreso > 150)?
val numAltos = viajesIngresosAltos.count()
println(s"Viajes con ingreso > 150: $numAltos")

// 2. ¿Cuál tiene mayor ingreso?
viajesIngresosAltos.show(1)

// 3. Sólo viajes de Madrid
viajes.filter($"ciudad" === "Madrid").show()

Viajes con ingreso > 150: 16
+-------+-------------+
| ciudad|ingreso_viaje|
+-------+-------------+
|Sevilla|        450.0|
+-------+-------------+
only showing top 1 row
+---+----------+------+------------+------+
| id|     fecha|ciudad|distancia_km|precio|
+---+----------+------+------------+------+
|  1|2026-05-01|Madrid|        12.5|  10.0|
|  3|2026-05-02|Madrid|        15.0|  12.0|
|  6|2026-05-03|Madrid|         5.0|   5.0|
|  8|2026-05-04|Madrid|        22.0|  16.0|
| 10|2026-05-05|Madrid|        14.0|  11.0|
| 12|2026-05-06|Madrid|        16.0|  13.0|
| 14|2026-05-07|Madrid|         9.0|   8.0|
| 17|2026-05-09|Madrid|        17.0|  14.0|
| 20|2026-05-10|Madrid|        21.0|  17.0|
| 22|2026-05-11|Madrid|        18.0|  15.0|
| 24|2026-05-12|Madrid|        20.0|  16.0|
| 26|2026-05-13|Madrid|        23.0|  18.0|
| 28|2026-05-14|Madrid|         7.0|   6.0|
| 30|2026-05-15|Madrid|        19.0|  15.0|
+---+----------+------+------------+------+



numAltos: Long = 16L

### 💡 Preguntas — Manipulación

In [25]:
// 1. Eliminar columna distancia
viajes.drop("distancia_km").show(5)

// 2. Renombrar precio → precio_unitario
viajes.withColumnRenamed("precio", "precio_unitario").show(5)

// 3. Quitar duplicados por ciudad (una fila por ciudad)
viajes.dropDuplicates("ciudad").show()

+---+----------+---------+------+
| id|     fecha|   ciudad|precio|
+---+----------+---------+------+
|  1|2026-05-01|   Madrid|  10.0|
|  2|2026-05-01|Barcelona|   7.0|
|  3|2026-05-02|   Madrid|  12.0|
|  4|2026-05-02| Valencia|   6.0|
|  5|2026-05-03|  Sevilla|  15.0|
+---+----------+---------+------+
only showing top 5 rows
+---+----------+---------+------------+---------------+
| id|     fecha|   ciudad|distancia_km|precio_unitario|
+---+----------+---------+------------+---------------+
|  1|2026-05-01|   Madrid|        12.5|           10.0|
|  2|2026-05-01|Barcelona|         8.0|            7.0|
|  3|2026-05-02|   Madrid|        15.0|           12.0|
|  4|2026-05-02| Valencia|         6.0|            6.0|
|  5|2026-05-03|  Sevilla|        20.0|           15.0|
+---+----------+---------+------------+---------------+
only showing top 5 rows
+---+----------+---------+------------+------+
| id|     fecha|   ciudad|distancia_km|precio|
+---+----------+---------+------------+------+
|

### 💡 Preguntas — Análisis

In [26]:
// 1. Ordenar por precio
viajes.orderBy($"precio".desc).show()

// 2. Cambiar filtro a > 100
viajes
  .withColumn("ingreso_viaje", $"distancia_km" * $"precio")
  .filter($"ingreso_viaje" > 100)
  .select("ciudad", "ingreso_viaje")
  .orderBy($"ingreso_viaje".desc)
  .show()

// 3. ¿Qué ciudad aparece más?
viajes.groupBy("ciudad").count().orderBy($"count".desc).show()

+---+----------+---------+------------+------+
| id|     fecha|   ciudad|distancia_km|precio|
+---+----------+---------+------------+------+
| 13|2026-05-07|  Sevilla|        25.0|  18.0|
| 26|2026-05-13|   Madrid|        23.0|  18.0|
| 20|2026-05-10|   Madrid|        21.0|  17.0|
|  8|2026-05-04|   Madrid|        22.0|  16.0|
| 24|2026-05-12|   Madrid|        20.0|  16.0|
| 22|2026-05-11|   Madrid|        18.0|  15.0|
|  5|2026-05-03|  Sevilla|        20.0|  15.0|
| 30|2026-05-15|   Madrid|        19.0|  15.0|
| 19|2026-05-10|  Sevilla|        19.0|  14.0|
| 17|2026-05-09|   Madrid|        17.0|  14.0|
|  7|2026-05-04|   Bilbao|        18.0|  13.0|
| 12|2026-05-06|   Madrid|        16.0|  13.0|
| 27|2026-05-14|  Sevilla|        18.0|  13.0|
|  3|2026-05-02|   Madrid|        15.0|  12.0|
| 10|2026-05-05|   Madrid|        14.0|  11.0|
| 25|2026-05-13|   Bilbao|        14.0|  11.0|
| 29|2026-05-15|Barcelona|        12.0|  10.0|
| 16|2026-05-08|   Bilbao|        13.0|  10.0|
|  1|2026-05-

### 💾 Guardar resultados — pipeline 1 a CSV

In [27]:
val resultadoViajes = viajes
  .withColumn("ingreso_viaje", $"distancia_km" * $"precio")
  .filter($"ingreso_viaje" > 150)
  .select("ciudad", "ingreso_viaje")
  .orderBy($"ingreso_viaje".desc)

resultadoViajes.write
  .mode("overwrite")
  .option("header", "true")
  .csv("salidas/viajes_ingresos_altos")

println("✅ Guardado en salidas/viajes_ingresos_altos")

✅ Guardado en salidas/viajes_ingresos_altos


resultadoViajes: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, ingreso_viaje: double]

### 🔹 Pipeline 2 — `precio_por_km`

1. `precio_por_km = precio / distancia_km`
2. Filtrar `precio_por_km > 0.8`
3. Seleccionar `ciudad`, `fecha`, `precio_por_km`
4. Ordenar `precio_por_km` descendente
5. Guardar en CSV


In [28]:
val viajesPrecioPorKm = viajes
  .withColumn("precio_por_km", $"precio" / $"distancia_km")
  .filter($"precio_por_km" > 0.8)
  .select("ciudad", "fecha", "precio_por_km")
  .orderBy($"precio_por_km".desc)

viajesPrecioPorKm.show()

viajesPrecioPorKm.write
  .mode("overwrite")
  .option("header", "true")
  .csv("salidas/viajes_precio_por_km")

println("✅ Guardado en salidas/viajes_precio_por_km")

+---------+----------+------------------+
|   ciudad|     fecha|     precio_por_km|
+---------+----------+------------------+
| Valencia|2026-05-02|               1.0|
|   Madrid|2026-05-03|               1.0|
| Valencia|2026-05-09|            0.9375|
| Valencia|2026-05-06|0.9285714285714286|
| Valencia|2026-05-12|0.9230769230769231|
|Barcelona|2026-05-05|               0.9|
|   Madrid|2026-05-07|0.8888888888888888|
|Barcelona|2026-05-11|0.8888888888888888|
|Barcelona|2026-05-01|             0.875|
|Barcelona|2026-05-08|0.8636363636363636|
|   Madrid|2026-05-14|0.8571428571428571|
|   Madrid|2026-05-11|0.8333333333333334|
|Barcelona|2026-05-15|0.8333333333333334|
|   Madrid|2026-05-09|0.8235294117647058|
|   Madrid|2026-05-06|            0.8125|
|   Madrid|2026-05-10|0.8095238095238095|
+---------+----------+------------------+

✅ Guardado en salidas/viajes_precio_por_km


viajesPrecioPorKm: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, fecha: string ... 1 more field]

---

# 🟣 Sesión 2 — Agregaciones, agrupaciones, ventanas, `rollup`, `cube` y `pivot`

## 0. Inicialización adicional (`Window`)


In [29]:
import org.apache.spark.sql.expressions.Window
println("✅ Window importado")

✅ Window importado


import org.apache.spark.sql.expressions.Window

## 📦 Dataset de práctica — `ventas` (mayo)

In [30]:
val ventasMayo = Seq(
  (1, "2026-05-01", "Madrid", "Tecnología", "Laptop", 1200.0, 2),
  (2, "2026-05-01", "Barcelona", "Tecnología", "Monitor", 350.0, 1),
  (3, "2026-05-01", "Valencia", "Oficina", "Silla", 120.0, 4),
  (4, "2026-05-02", "Madrid", "Tecnología", "Teclado", 45.0, 6),
  (5, "2026-05-02", "Sevilla", "Oficina", "Mesa", 250.0, 2),
  (6, "2026-05-02", "Bilbao", "Tecnología", "Tablet", 600.0, 1),
  (7, "2026-05-03", "Madrid", "Tecnología", "Monitor", 350.0, 2),
  (8, "2026-05-03", "Barcelona", "Oficina", "Silla", 120.0, 3),
  (9, "2026-05-03", "Valencia", "Tecnología", "Ratón", 25.0, 10),
  (10, "2026-05-04", "Madrid", "Oficina", "Mesa", 250.0, 1),
  (11, "2026-05-04", "Sevilla", "Tecnología", "Laptop", 1200.0, 1),
  (12, "2026-05-04", "Bilbao", "Tecnología", "Teclado", 45.0, 5),
  (13, "2026-05-05", "Madrid", "Tecnología", "Tablet", 600.0, 2),
  (14, "2026-05-05", "Barcelona", "Tecnología", "Laptop", 1200.0, 1),
  (15, "2026-05-05", "Valencia", "Oficina", "Archivador", 80.0, 4),
  (16, "2026-05-06", "Madrid", "Tecnología", "Ratón", 25.0, 12),
  (17, "2026-05-06", "Sevilla", "Tecnología", "Monitor", 350.0, 1),
  (18, "2026-05-06", "Bilbao", "Oficina", "Silla", 120.0, 2),
  (19, "2026-05-07", "Madrid", "Oficina", "Silla", 120.0, 5),
  (20, "2026-05-07", "Barcelona", "Tecnología", "Tablet", 600.0, 2),
  (21, "2026-05-07", "Valencia", "Tecnología", "Teclado", 45.0, 7),
  (22, "2026-05-08", "Madrid", "Tecnología", "Laptop", 1200.0, 1),
  (23, "2026-05-08", "Sevilla", "Oficina", "Mesa", 250.0, 1),
  (24, "2026-05-08", "Bilbao", "Tecnología", "Monitor", 350.0, 2),
  (25, "2026-05-09", "Madrid", "Tecnología", "Monitor", 350.0, 3),
  (26, "2026-05-09", "Barcelona", "Tecnología", "Ratón", 25.0, 15),
  (27, "2026-05-09", "Valencia", "Oficina", "Mesa", 250.0, 2),
  (28, "2026-05-10", "Madrid", "Oficina", "Archivador", 80.0, 6),
  (29, "2026-05-10", "Sevilla", "Tecnología", "Tablet", 600.0, 1),
  (30, "2026-05-10", "Bilbao", "Tecnología", "Laptop", 1200.0, 1),
  (31, "2026-05-11", "Madrid", "Tecnología", "Teclado", 45.0, 8),
  (32, "2026-05-11", "Barcelona", "Oficina", "Mesa", 250.0, 1),
  (33, "2026-05-11", "Valencia", "Tecnología", "Monitor", 350.0, 1),
  (34, "2026-05-12", "Madrid", "Tecnología", "Laptop", 1200.0, 3),
  (35, "2026-05-12", "Sevilla", "Oficina", "Silla", 120.0, 4),
  (36, "2026-05-12", "Bilbao", "Tecnología", "Ratón", 25.0, 9),
  (37, "2026-05-13", "Madrid", "Oficina", "Mesa", 250.0, 2),
  (38, "2026-05-13", "Barcelona", "Tecnología", "Monitor", 350.0, 2),
  (39, "2026-05-13", "Valencia", "Tecnología", "Tablet", 600.0, 1),
  (40, "2026-05-14", "Madrid", "Tecnología", "Laptop", 1200.0, 2),
  (41, "2026-05-14", "Sevilla", "Tecnología", "Teclado", 45.0, 6),
  (42, "2026-05-14", "Bilbao", "Oficina", "Archivador", 80.0, 3),
  (43, "2026-05-15", "Madrid", "Tecnología", "Tablet", 600.0, 1),
  (44, "2026-05-15", "Barcelona", "Oficina", "Silla", 120.0, 2),
  (45, "2026-05-15", "Valencia", "Tecnología", "Laptop", 1200.0, 1)
).toDF("id", "fecha", "ciudad", "categoria", "producto", "precio_unitario", "cantidad")

val ventas = ventasMayo
  .withColumn("total", $"precio_unitario" * $"cantidad")

ventas.show(10)

+---+----------+---------+----------+--------+---------------+--------+------+
| id|     fecha|   ciudad| categoria|producto|precio_unitario|cantidad| total|
+---+----------+---------+----------+--------+---------------+--------+------+
|  1|2026-05-01|   Madrid|Tecnología|  Laptop|         1200.0|       2|2400.0|
|  2|2026-05-01|Barcelona|Tecnología| Monitor|          350.0|       1| 350.0|
|  3|2026-05-01| Valencia|   Oficina|   Silla|          120.0|       4| 480.0|
|  4|2026-05-02|   Madrid|Tecnología| Teclado|           45.0|       6| 270.0|
|  5|2026-05-02|  Sevilla|   Oficina|    Mesa|          250.0|       2| 500.0|
|  6|2026-05-02|   Bilbao|Tecnología|  Tablet|          600.0|       1| 600.0|
|  7|2026-05-03|   Madrid|Tecnología| Monitor|          350.0|       2| 700.0|
|  8|2026-05-03|Barcelona|   Oficina|   Silla|          120.0|       3| 360.0|
|  9|2026-05-03| Valencia|Tecnología|   Ratón|           25.0|      10| 250.0|
| 10|2026-05-04|   Madrid|   Oficina|    Mesa|      

ventasMayo: org.apache.spark.sql.package.DataFrame = [id: int, fecha: string ... 5 more fields]
ventas: org.apache.spark.sql.package.DataFrame = [id: int, fecha: string ... 6 more fields]

## 🧪 Bloque 1 — Exploración inicial

### Ejercicio 1 — Ver primeras filas

In [31]:
ventas.show(5)

+---+----------+---------+----------+--------+---------------+--------+------+
| id|     fecha|   ciudad| categoria|producto|precio_unitario|cantidad| total|
+---+----------+---------+----------+--------+---------------+--------+------+
|  1|2026-05-01|   Madrid|Tecnología|  Laptop|         1200.0|       2|2400.0|
|  2|2026-05-01|Barcelona|Tecnología| Monitor|          350.0|       1| 350.0|
|  3|2026-05-01| Valencia|   Oficina|   Silla|          120.0|       4| 480.0|
|  4|2026-05-02|   Madrid|Tecnología| Teclado|           45.0|       6| 270.0|
|  5|2026-05-02|  Sevilla|   Oficina|    Mesa|          250.0|       2| 500.0|
+---+----------+---------+----------+--------+---------------+--------+------+
only showing top 5 rows


### Ejercicio 2 — Revisar schema

In [32]:
ventas.printSchema()

root
 |-- id: integer (nullable = false)
 |-- fecha: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- precio_unitario: double (nullable = false)
 |-- cantidad: integer (nullable = false)
 |-- total: double (nullable = false)



### Ejercicio 3 — Contar registros

In [33]:
ventas.count()

res33: Long = 45L

### Ejercicio 4 — Estadísticas básicas

In [34]:
ventas.describe("precio_unitario", "cantidad", "total").show()

+-------+------------------+------------------+-----------------+
|summary|   precio_unitario|          cantidad|            total|
+-------+------------------+------------------+-----------------+
|  count|                45|                45|               45|
|   mean| 409.6666666666667|3.3777777777777778|            704.0|
| stddev|412.26977046148266| 3.214236010518383|670.8166664596222|
|    min|              25.0|                 1|            225.0|
|    max|            1200.0|                15|           3600.0|
+-------+------------------+------------------+-----------------+



## 🧪 Bloque 2 — `groupBy()` y `count()`

### Ejercicio 5 — Número de ventas por ciudad

In [35]:
val ventasPorCiudad = ventas
  .groupBy("ciudad")
  .count()
  .orderBy($"count".desc)

ventasPorCiudad.show()

+---------+-----+
|   ciudad|count|
+---------+-----+
|   Madrid|   15|
|Barcelona|    8|
| Valencia|    8|
|  Sevilla|    7|
|   Bilbao|    7|
+---------+-----+



ventasPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, count: bigint]

### Ejercicio 6 — Número de ventas por categoría

In [36]:
val ventasPorCategoria = ventas
  .groupBy("categoria")
  .count()
  .orderBy($"count".desc)

ventasPorCategoria.show()

+----------+-----+
| categoria|count|
+----------+-----+
|Tecnología|   30|
|   Oficina|   15|
+----------+-----+



ventasPorCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string, count: bigint]

### Ejercicio 7 — Número de ventas por producto

In [37]:
val ventasPorProducto = ventas
  .groupBy("producto")
  .count()
  .orderBy($"count".desc)

ventasPorProducto.show()

+----------+-----+
|  producto|count|
+----------+-----+
|    Laptop|    8|
|   Monitor|    7|
|     Silla|    6|
|      Mesa|    6|
|    Tablet|    6|
|   Teclado|    5|
|     Ratón|    4|
|Archivador|    3|
+----------+-----+



ventasPorProducto: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [producto: string, count: bigint]

## 🧪 Bloque 3 — `sum`, `avg`, `min`, `max`

### Ejercicio 8 — Total facturado por ciudad

In [38]:
val facturacionPorCiudad = ventas
  .groupBy("ciudad")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"total_facturado".desc)

facturacionPorCiudad.show()

+---------+---------------+
|   ciudad|total_facturado|
+---------+---------------+
|   Madrid|        15910.0|
|Barcelona|         4675.0|
| Valencia|         4015.0|
|  Sevilla|         3650.0|
|   Bilbao|         3430.0|
+---------+---------------+



facturacionPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, total_facturado: double]

### Ejercicio 9 — Total facturado por categoría

In [39]:
val facturacionPorCategoria = ventas
  .groupBy("categoria")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"total_facturado".desc)

facturacionPorCategoria.show()

+----------+---------------+
| categoria|total_facturado|
+----------+---------------+
|Tecnología|        25990.0|
|   Oficina|         5690.0|
+----------+---------------+



facturacionPorCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string, total_facturado: double]

### Ejercicio 10 — Ticket medio por ciudad

In [40]:
val ticketMedioPorCiudad = ventas
  .groupBy("ciudad")
  .agg(avg("total").as("ticket_medio"))
  .orderBy($"ticket_medio".desc)

ticketMedioPorCiudad.show()

+---------+------------------+
|   ciudad|      ticket_medio|
+---------+------------------+
|   Madrid|1060.6666666666667|
|Barcelona|           584.375|
|  Sevilla| 521.4285714285714|
| Valencia|           501.875|
|   Bilbao|             490.0|
+---------+------------------+



ticketMedioPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, ticket_medio: double]

### Ejercicio 11 — Venta mínima y máxima por ciudad

In [41]:
val minMaxPorCiudad = ventas
  .groupBy("ciudad")
  .agg(
    min("total").as("venta_minima"),
    max("total").as("venta_maxima")
  )
  .orderBy($"venta_maxima".desc)

minMaxPorCiudad.show()

+---------+------------+------------+
|   ciudad|venta_minima|venta_maxima|
+---------+------------+------------+
|   Madrid|       250.0|      3600.0|
|Barcelona|       240.0|      1200.0|
| Valencia|       250.0|      1200.0|
|  Sevilla|       250.0|      1200.0|
|   Bilbao|       225.0|      1200.0|
+---------+------------+------------+



minMaxPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, venta_minima: double ... 1 more field]

## 🧪 Bloque 4 — `agg()` con varias métricas

### Ejercicio 12 — Resumen completo por ciudad

In [42]:
val resumenCiudad = ventas
  .groupBy("ciudad")
  .agg(
    count("id").as("num_ventas"),
    sum("cantidad").as("unidades_vendidas"),
    sum("total").as("total_facturado"),
    avg("total").as("ticket_medio"),
    max("total").as("venta_maxima")
  )
  .orderBy($"total_facturado".desc)

resumenCiudad.show()

+---------+----------+-----------------+---------------+------------------+------------+
|   ciudad|num_ventas|unidades_vendidas|total_facturado|      ticket_medio|venta_maxima|
+---------+----------+-----------------+---------------+------------------+------------+
|   Madrid|        15|               56|        15910.0|1060.6666666666667|      3600.0|
|Barcelona|         8|               27|         4675.0|           584.375|      1200.0|
| Valencia|         8|               30|         4015.0|           501.875|      1200.0|
|  Sevilla|         7|               16|         3650.0| 521.4285714285714|      1200.0|
|   Bilbao|         7|               23|         3430.0|             490.0|      1200.0|
+---------+----------+-----------------+---------------+------------------+------------+



resumenCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, num_ventas: bigint ... 4 more fields]

### Ejercicio 13 — Resumen completo por producto

In [43]:
val resumenProducto = ventas
  .groupBy("producto")
  .agg(
    count("id").as("num_ventas"),
    sum("cantidad").as("unidades_vendidas"),
    sum("total").as("total_facturado"),
    avg("total").as("ticket_medio"),
    max("total").as("venta_maxima")
  )
  .orderBy($"total_facturado".desc)

resumenProducto.show()

+----------+----------+-----------------+---------------+-----------------+------------+
|  producto|num_ventas|unidades_vendidas|total_facturado|     ticket_medio|venta_maxima|
+----------+----------+-----------------+---------------+-----------------+------------+
|    Laptop|         8|               12|        14400.0|           1800.0|      3600.0|
|    Tablet|         6|                8|         4800.0|            800.0|      1200.0|
|   Monitor|         7|               12|         4200.0|            600.0|      1050.0|
|     Silla|         6|               20|         2400.0|            400.0|       600.0|
|      Mesa|         6|                9|         2250.0|            375.0|       500.0|
|   Teclado|         5|               32|         1440.0|            288.0|       360.0|
|     Ratón|         4|               46|         1150.0|            287.5|       375.0|
|Archivador|         3|               13|         1040.0|346.6666666666667|       480.0|
+----------+---------

resumenProducto: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [producto: string, num_ventas: bigint ... 4 more fields]

### Ejercicio 14 — Resumen por ciudad y categoría

In [44]:
val resumenCiudadCategoria = ventas
  .groupBy("ciudad", "categoria")
  .agg(
    count("id").as("num_ventas"),
    sum("total").as("total_facturado")
  )
  .orderBy($"ciudad", $"total_facturado".desc)

resumenCiudadCategoria.show()

+---------+----------+----------+---------------+
|   ciudad| categoria|num_ventas|total_facturado|
+---------+----------+----------+---------------+
|Barcelona|Tecnología|         5|         3825.0|
|Barcelona|   Oficina|         3|          850.0|
|   Bilbao|Tecnología|         5|         2950.0|
|   Bilbao|   Oficina|         2|          480.0|
|   Madrid|Tecnología|        11|        14080.0|
|   Madrid|   Oficina|         4|         1830.0|
|  Sevilla|Tecnología|         4|         2420.0|
|  Sevilla|   Oficina|         3|         1230.0|
| Valencia|Tecnología|         5|         2715.0|
| Valencia|   Oficina|         3|         1300.0|
+---------+----------+----------+---------------+



resumenCiudadCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, categoria: string ... 2 more fields]

## 🧪 Bloque 5 — Ranking con funciones de ventana

### Ejercicio 15 — Ranking de ventas dentro de cada ciudad

In [45]:
val ventanaCiudad = Window
  .partitionBy("ciudad")
  .orderBy($"total".desc)

val rankingVentasCiudad = ventas
  .withColumn("ranking_ciudad", row_number().over(ventanaCiudad))
  .orderBy($"ciudad", $"ranking_ciudad")

rankingVentasCiudad.show(50)

+---+----------+---------+----------+----------+---------------+--------+------+--------------+
| id|     fecha|   ciudad| categoria|  producto|precio_unitario|cantidad| total|ranking_ciudad|
+---+----------+---------+----------+----------+---------------+--------+------+--------------+
| 14|2026-05-05|Barcelona|Tecnología|    Laptop|         1200.0|       1|1200.0|             1|
| 20|2026-05-07|Barcelona|Tecnología|    Tablet|          600.0|       2|1200.0|             2|
| 38|2026-05-13|Barcelona|Tecnología|   Monitor|          350.0|       2| 700.0|             3|
| 26|2026-05-09|Barcelona|Tecnología|     Ratón|           25.0|      15| 375.0|             4|
|  8|2026-05-03|Barcelona|   Oficina|     Silla|          120.0|       3| 360.0|             5|
|  2|2026-05-01|Barcelona|Tecnología|   Monitor|          350.0|       1| 350.0|             6|
| 32|2026-05-11|Barcelona|   Oficina|      Mesa|          250.0|       1| 250.0|             7|
| 44|2026-05-15|Barcelona|   Oficina|   

ventanaCiudad: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@e2c4c6e
rankingVentasCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: int, fecha: string ... 7 more fields]

### Ejercicio 16 — Top 1 venta de cada ciudad

In [46]:
val topVentaPorCiudad = rankingVentasCiudad
  .filter($"ranking_ciudad" === 1)
  .select("ciudad", "producto", "total", "ranking_ciudad")
  .orderBy($"total".desc)

topVentaPorCiudad.show()

+---------+--------+------+--------------+
|   ciudad|producto| total|ranking_ciudad|
+---------+--------+------+--------------+
|   Madrid|  Laptop|3600.0|             1|
|Barcelona|  Laptop|1200.0|             1|
|   Bilbao|  Laptop|1200.0|             1|
|  Sevilla|  Laptop|1200.0|             1|
| Valencia|  Laptop|1200.0|             1|
+---------+--------+------+--------------+



topVentaPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, producto: string ... 2 more fields]

### Ejercicio 17 — Top 3 ventas de cada ciudad

In [47]:
val top3VentasPorCiudad = rankingVentasCiudad
  .filter($"ranking_ciudad" <= 3)
  .select("ciudad", "producto", "total", "ranking_ciudad")
  .orderBy($"ciudad", $"ranking_ciudad")

top3VentasPorCiudad.show(50)

+---------+--------+------+--------------+
|   ciudad|producto| total|ranking_ciudad|
+---------+--------+------+--------------+
|Barcelona|  Laptop|1200.0|             1|
|Barcelona|  Tablet|1200.0|             2|
|Barcelona| Monitor| 700.0|             3|
|   Bilbao|  Laptop|1200.0|             1|
|   Bilbao| Monitor| 700.0|             2|
|   Bilbao|  Tablet| 600.0|             3|
|   Madrid|  Laptop|3600.0|             1|
|   Madrid|  Laptop|2400.0|             2|
|   Madrid|  Laptop|2400.0|             3|
|  Sevilla|  Laptop|1200.0|             1|
|  Sevilla|  Tablet| 600.0|             2|
|  Sevilla|    Mesa| 500.0|             3|
| Valencia|  Laptop|1200.0|             1|
| Valencia|  Tablet| 600.0|             2|
| Valencia|    Mesa| 500.0|             3|
+---------+--------+------+--------------+



top3VentasPorCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, producto: string ... 2 more fields]

## 🧪 Bloque 6 — `rollup()`

### Ejercicio 18 — Subtotales por ciudad y categoría

In [48]:
val rollupCiudadCategoria = ventas
  .rollup("ciudad", "categoria")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"ciudad", $"categoria")

rollupCiudadCategoria.show(50)

+---------+----------+---------------+
|   ciudad| categoria|total_facturado|
+---------+----------+---------------+
|     NULL|      NULL|        31680.0|
|Barcelona|      NULL|         4675.0|
|Barcelona|   Oficina|          850.0|
|Barcelona|Tecnología|         3825.0|
|   Bilbao|      NULL|         3430.0|
|   Bilbao|   Oficina|          480.0|
|   Bilbao|Tecnología|         2950.0|
|   Madrid|      NULL|        15910.0|
|   Madrid|   Oficina|         1830.0|
|   Madrid|Tecnología|        14080.0|
|  Sevilla|      NULL|         3650.0|
|  Sevilla|   Oficina|         1230.0|
|  Sevilla|Tecnología|         2420.0|
| Valencia|      NULL|         4015.0|
| Valencia|   Oficina|         1300.0|
| Valencia|Tecnología|         2715.0|
+---------+----------+---------------+



rollupCiudadCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, categoria: string ... 1 more field]

## 🧪 Bloque 7 — `cube()`

### Ejercicio 19 — Análisis multidimensional por ciudad y categoría

In [49]:
val cubeCiudadCategoria = ventas
  .cube("ciudad", "categoria")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"ciudad", $"categoria")

cubeCiudadCategoria.show(50)

+---------+----------+---------------+
|   ciudad| categoria|total_facturado|
+---------+----------+---------------+
|     NULL|      NULL|        31680.0|
|     NULL|   Oficina|         5690.0|
|     NULL|Tecnología|        25990.0|
|Barcelona|      NULL|         4675.0|
|Barcelona|   Oficina|          850.0|
|Barcelona|Tecnología|         3825.0|
|   Bilbao|      NULL|         3430.0|
|   Bilbao|   Oficina|          480.0|
|   Bilbao|Tecnología|         2950.0|
|   Madrid|      NULL|        15910.0|
|   Madrid|   Oficina|         1830.0|
|   Madrid|Tecnología|        14080.0|
|  Sevilla|      NULL|         3650.0|
|  Sevilla|   Oficina|         1230.0|
|  Sevilla|Tecnología|         2420.0|
| Valencia|      NULL|         4015.0|
| Valencia|   Oficina|         1300.0|
| Valencia|Tecnología|         2715.0|
+---------+----------+---------------+



cubeCiudadCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, categoria: string ... 1 more field]

## 🧪 Bloque 8 — `pivot()`

### Ejercicio 20 — Tabla dinámica facturación ciudad × categoría

In [50]:
val pivotCiudadCategoria = ventas
  .groupBy("ciudad")
  .pivot("categoria")
  .agg(sum("total"))
  .orderBy("ciudad")

pivotCiudadCategoria.show()

+---------+-------+----------+
|   ciudad|Oficina|Tecnología|
+---------+-------+----------+
|Barcelona|  850.0|    3825.0|
|   Bilbao|  480.0|    2950.0|
|   Madrid| 1830.0|   14080.0|
|  Sevilla| 1230.0|    2420.0|
| Valencia| 1300.0|    2715.0|
+---------+-------+----------+



pivotCiudadCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, Oficina: double ... 1 more field]

### Ejercicio 21 — Pivot ciudad × producto

In [51]:
val pivotCiudadProducto = ventas
  .groupBy("ciudad")
  .pivot("producto")
  .agg(sum("total"))
  .orderBy("ciudad")

pivotCiudadProducto.show()

+---------+----------+------+-----+-------+-----+-----+------+-------+
|   ciudad|Archivador|Laptop| Mesa|Monitor|Ratón|Silla|Tablet|Teclado|
+---------+----------+------+-----+-------+-----+-----+------+-------+
|Barcelona|      NULL|1200.0|250.0| 1050.0|375.0|600.0|1200.0|   NULL|
|   Bilbao|     240.0|1200.0| NULL|  700.0|225.0|240.0| 600.0|  225.0|
|   Madrid|     480.0|9600.0|750.0| 1750.0|300.0|600.0|1800.0|  630.0|
|  Sevilla|      NULL|1200.0|750.0|  350.0| NULL|480.0| 600.0|  270.0|
| Valencia|     320.0|1200.0|500.0|  350.0|250.0|480.0| 600.0|  315.0|
+---------+----------+------+-----+-------+-----+-----+------+-------+



pivotCiudadProducto: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, Archivador: double ... 7 more fields]

---

## 🏢 Caso de Estudio Propuesto — *MarketNova Analytics*

**MarketNova** es una cadena de tiendas online. Durante mayo registra ventas en varias ciudades. El equipo directivo necesita responder:

- ¿Qué **ciudad** genera más ingresos?
- ¿Qué **categoría** es más rentable?
- ¿Qué **productos** lideran las ventas?
- ¿Cuáles son las **mejores ventas** dentro de cada ciudad?
- ¿Cómo se distribuye la facturación por ciudad y categoría?

### 📥 Dataset


In [52]:
val marketNovaRaw = Seq(
  (1, "2026-05-01", "Madrid", "Electrónica", "Portátil", 950.0, 2),
  (2, "2026-05-01", "Madrid", "Hogar", "Aspiradora", 180.0, 1),
  (3, "2026-05-02", "Barcelona", "Electrónica", "Tablet", 420.0, 3),
  (4, "2026-05-02", "Valencia", "Deporte", "Bicicleta", 300.0, 2),
  (5, "2026-05-03", "Sevilla", "Hogar", "Cafetera", 90.0, 4),
  (6, "2026-05-03", "Bilbao", "Electrónica", "Auriculares", 75.0, 6),
  (7, "2026-05-04", "Madrid", "Deporte", "Cinta correr", 650.0, 1),
  (8, "2026-05-04", "Barcelona", "Hogar", "Microondas", 140.0, 2),
  (9, "2026-05-05", "Valencia", "Electrónica", "Monitor", 280.0, 2),
  (10, "2026-05-05", "Sevilla", "Deporte", "Mancuernas", 45.0, 8),
  (11, "2026-05-06", "Bilbao", "Hogar", "Batidora", 60.0, 5),
  (12, "2026-05-06", "Madrid", "Electrónica", "Smartphone", 700.0, 2),
  (13, "2026-05-07", "Barcelona", "Deporte", "Bicicleta", 300.0, 1),
  (14, "2026-05-07", "Valencia", "Hogar", "Cafetera", 90.0, 3),
  (15, "2026-05-08", "Sevilla", "Electrónica", "Tablet", 420.0, 2),
  (16, "2026-05-08", "Bilbao", "Deporte", "Balón", 25.0, 10),
  (17, "2026-05-09", "Madrid", "Hogar", "Microondas", 140.0, 3),
  (18, "2026-05-09", "Barcelona", "Electrónica", "Portátil", 950.0, 1),
  (19, "2026-05-10", "Valencia", "Deporte", "Cinta correr", 650.0, 1),
  (20, "2026-05-10", "Sevilla", "Hogar", "Aspiradora", 180.0, 2),
  (21, "2026-05-11", "Bilbao", "Electrónica", "Smartphone", 700.0, 1),
  (22, "2026-05-11", "Madrid", "Deporte", "Bicicleta", 300.0, 2),
  (23, "2026-05-12", "Barcelona", "Hogar", "Cafetera", 90.0, 5),
  (24, "2026-05-12", "Valencia", "Electrónica", "Auriculares", 75.0, 7),
  (25, "2026-05-13", "Sevilla", "Deporte", "Cinta correr", 650.0, 1),
  (26, "2026-05-13", "Bilbao", "Hogar", "Microondas", 140.0, 2),
  (27, "2026-05-14", "Madrid", "Electrónica", "Monitor", 280.0, 4),
  (28, "2026-05-14", "Barcelona", "Deporte", "Mancuernas", 45.0, 6),
  (29, "2026-05-15", "Valencia", "Hogar", "Aspiradora", 180.0, 2),
  (30, "2026-05-15", "Sevilla", "Electrónica", "Smartphone", 700.0, 1),
  (31, "2026-05-16", "Bilbao", "Deporte", "Bicicleta", 300.0, 2),
  (32, "2026-05-16", "Madrid", "Hogar", "Batidora", 60.0, 5),
  (33, "2026-05-17", "Barcelona", "Electrónica", "Monitor", 280.0, 3),
  (34, "2026-05-17", "Valencia", "Deporte", "Balón", 25.0, 12),
  (35, "2026-05-18", "Sevilla", "Hogar", "Microondas", 140.0, 2),
  (36, "2026-05-18", "Bilbao", "Electrónica", "Portátil", 950.0, 1)
).toDF("id", "fecha", "ciudad", "categoria", "producto", "precio_unitario", "cantidad")

val marketNova = marketNovaRaw
  .withColumn("total", $"precio_unitario" * $"cantidad")

marketNova.show(10)

+---+----------+---------+-----------+------------+---------------+--------+------+
| id|     fecha|   ciudad|  categoria|    producto|precio_unitario|cantidad| total|
+---+----------+---------+-----------+------------+---------------+--------+------+
|  1|2026-05-01|   Madrid|Electrónica|    Portátil|          950.0|       2|1900.0|
|  2|2026-05-01|   Madrid|      Hogar|  Aspiradora|          180.0|       1| 180.0|
|  3|2026-05-02|Barcelona|Electrónica|      Tablet|          420.0|       3|1260.0|
|  4|2026-05-02| Valencia|    Deporte|   Bicicleta|          300.0|       2| 600.0|
|  5|2026-05-03|  Sevilla|      Hogar|    Cafetera|           90.0|       4| 360.0|
|  6|2026-05-03|   Bilbao|Electrónica| Auriculares|           75.0|       6| 450.0|
|  7|2026-05-04|   Madrid|    Deporte|Cinta correr|          650.0|       1| 650.0|
|  8|2026-05-04|Barcelona|      Hogar|  Microondas|          140.0|       2| 280.0|
|  9|2026-05-05| Valencia|Electrónica|     Monitor|          280.0|       2|

marketNovaRaw: org.apache.spark.sql.package.DataFrame = [id: int, fecha: string ... 5 more fields]
marketNova: org.apache.spark.sql.package.DataFrame = [id: int, fecha: string ... 6 more fields]

### 🎯 Pregunta 1 — Exploración inicial

Primeras 10 filas, schema y estadísticas de `precio_unitario`, `cantidad` y `total`.


In [53]:
marketNova.show(10)
marketNova.printSchema()
marketNova.describe("precio_unitario", "cantidad", "total").show()

+---+----------+---------+-----------+------------+---------------+--------+------+
| id|     fecha|   ciudad|  categoria|    producto|precio_unitario|cantidad| total|
+---+----------+---------+-----------+------------+---------------+--------+------+
|  1|2026-05-01|   Madrid|Electrónica|    Portátil|          950.0|       2|1900.0|
|  2|2026-05-01|   Madrid|      Hogar|  Aspiradora|          180.0|       1| 180.0|
|  3|2026-05-02|Barcelona|Electrónica|      Tablet|          420.0|       3|1260.0|
|  4|2026-05-02| Valencia|    Deporte|   Bicicleta|          300.0|       2| 600.0|
|  5|2026-05-03|  Sevilla|      Hogar|    Cafetera|           90.0|       4| 360.0|
|  6|2026-05-03|   Bilbao|Electrónica| Auriculares|           75.0|       6| 450.0|
|  7|2026-05-04|   Madrid|    Deporte|Cinta correr|          650.0|       1| 650.0|
|  8|2026-05-04|Barcelona|      Hogar|  Microondas|          140.0|       2| 280.0|
|  9|2026-05-05| Valencia|Electrónica|     Monitor|          280.0|       2|

### 🎯 Pregunta 2 — Ventas por ciudad

In [54]:
marketNova
  .groupBy("ciudad")
  .count()
  .orderBy($"count".desc)
  .show()

+---------+-----+
|   ciudad|count|
+---------+-----+
|   Madrid|    8|
|Barcelona|    7|
| Valencia|    7|
|   Bilbao|    7|
|  Sevilla|    7|
+---------+-----+



### 🎯 Pregunta 3 — Facturación por ciudad

In [55]:
marketNova
  .groupBy("ciudad")
  .agg(sum("total").as("facturacion"))
  .orderBy($"facturacion".desc)
  .show()

+---------+-----------+
|   ciudad|facturacion|
+---------+-----------+
|   Madrid|     6570.0|
|Barcelona|     4350.0|
|  Sevilla|     3550.0|
|   Bilbao|     3530.0|
| Valencia|     3265.0|
+---------+-----------+



### 🎯 Pregunta 4 — Facturación por categoría

In [56]:
marketNova
  .groupBy("categoria")
  .agg(sum("total").as("facturacion"))
  .orderBy($"facturacion".desc)
  .show()

+-----------+-----------+
|  categoria|facturacion|
+-----------+-----------+
|Electrónica|    12195.0|
|    Deporte|     5230.0|
|      Hogar|     3840.0|
+-----------+-----------+



### 🎯 Pregunta 5 — Resumen por producto

In [57]:
marketNova
  .groupBy("producto")
  .agg(
    count("id").as("num_ventas"),
    sum("cantidad").as("unidades_vendidas"),
    sum("total").as("total_facturado"),
    avg("total").as("ticket_medio"),
    max("total").as("venta_maxima")
  )
  .orderBy($"total_facturado".desc)
  .show()

+------------+----------+-----------------+---------------+------------------+------------+
|    producto|num_ventas|unidades_vendidas|total_facturado|      ticket_medio|venta_maxima|
+------------+----------+-----------------+---------------+------------------+------------+
|    Portátil|         3|                4|         3800.0|1266.6666666666667|      1900.0|
|  Smartphone|         3|                4|         2800.0| 933.3333333333334|      1400.0|
|     Monitor|         3|                9|         2520.0|             840.0|      1120.0|
|   Bicicleta|         4|                7|         2100.0|             525.0|       600.0|
|      Tablet|         2|                5|         2100.0|            1050.0|      1260.0|
|Cinta correr|         3|                3|         1950.0|             650.0|       650.0|
|  Microondas|         4|                9|         1260.0|             315.0|       420.0|
|    Cafetera|         3|               12|         1080.0|             360.0|  

### 🎯 Pregunta 6 — Resumen por ciudad y categoría

In [58]:
marketNova
  .groupBy("ciudad", "categoria")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"ciudad", $"total_facturado".desc)
  .show(50)

+---------+-----------+---------------+
|   ciudad|  categoria|total_facturado|
+---------+-----------+---------------+
|Barcelona|Electrónica|         3050.0|
|Barcelona|      Hogar|          730.0|
|Barcelona|    Deporte|          570.0|
|   Bilbao|Electrónica|         2100.0|
|   Bilbao|    Deporte|          850.0|
|   Bilbao|      Hogar|          580.0|
|   Madrid|Electrónica|         4420.0|
|   Madrid|    Deporte|         1250.0|
|   Madrid|      Hogar|          900.0|
|  Sevilla|Electrónica|         1540.0|
|  Sevilla|    Deporte|         1010.0|
|  Sevilla|      Hogar|         1000.0|
| Valencia|    Deporte|         1550.0|
| Valencia|Electrónica|         1085.0|
| Valencia|      Hogar|          630.0|
+---------+-----------+---------------+



### 🎯 Pregunta 7 — Ranking de ventas por ciudad

In [59]:
val ventanaMN = Window.partitionBy("ciudad").orderBy($"total".desc)

val rankingMN = marketNova
  .withColumn("ranking", row_number().over(ventanaMN))
  .orderBy($"ciudad", $"ranking")

rankingMN.show(50)

+---+----------+---------+-----------+------------+---------------+--------+------+-------+
| id|     fecha|   ciudad|  categoria|    producto|precio_unitario|cantidad| total|ranking|
+---+----------+---------+-----------+------------+---------------+--------+------+-------+
|  3|2026-05-02|Barcelona|Electrónica|      Tablet|          420.0|       3|1260.0|      1|
| 18|2026-05-09|Barcelona|Electrónica|    Portátil|          950.0|       1| 950.0|      2|
| 33|2026-05-17|Barcelona|Electrónica|     Monitor|          280.0|       3| 840.0|      3|
| 23|2026-05-12|Barcelona|      Hogar|    Cafetera|           90.0|       5| 450.0|      4|
| 13|2026-05-07|Barcelona|    Deporte|   Bicicleta|          300.0|       1| 300.0|      5|
|  8|2026-05-04|Barcelona|      Hogar|  Microondas|          140.0|       2| 280.0|      6|
| 28|2026-05-14|Barcelona|    Deporte|  Mancuernas|           45.0|       6| 270.0|      7|
| 36|2026-05-18|   Bilbao|Electrónica|    Portátil|          950.0|       1| 950

ventanaMN: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@f5c2d83
rankingMN: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: int, fecha: string ... 7 more fields]

### 🎯 Pregunta 8 — Mejor venta de cada ciudad

In [60]:
rankingMN
  .filter($"ranking" === 1)
  .select("ciudad", "producto", "total", "ranking")
  .orderBy($"total".desc)
  .show()

+---------+------------+------+-------+
|   ciudad|    producto| total|ranking|
+---------+------------+------+-------+
|   Madrid|    Portátil|1900.0|      1|
|Barcelona|      Tablet|1260.0|      1|
|   Bilbao|    Portátil| 950.0|      1|
|  Sevilla|      Tablet| 840.0|      1|
| Valencia|Cinta correr| 650.0|      1|
+---------+------------+------+-------+



### 🎯 Pregunta 9 — Subtotales con `rollup`

In [61]:
marketNova
  .rollup("ciudad", "categoria")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"ciudad", $"categoria")
  .show(50)

+---------+-----------+---------------+
|   ciudad|  categoria|total_facturado|
+---------+-----------+---------------+
|     NULL|       NULL|        21265.0|
|Barcelona|       NULL|         4350.0|
|Barcelona|    Deporte|          570.0|
|Barcelona|Electrónica|         3050.0|
|Barcelona|      Hogar|          730.0|
|   Bilbao|       NULL|         3530.0|
|   Bilbao|    Deporte|          850.0|
|   Bilbao|Electrónica|         2100.0|
|   Bilbao|      Hogar|          580.0|
|   Madrid|       NULL|         6570.0|
|   Madrid|    Deporte|         1250.0|
|   Madrid|Electrónica|         4420.0|
|   Madrid|      Hogar|          900.0|
|  Sevilla|       NULL|         3550.0|
|  Sevilla|    Deporte|         1010.0|
|  Sevilla|Electrónica|         1540.0|
|  Sevilla|      Hogar|         1000.0|
| Valencia|       NULL|         3265.0|
| Valencia|    Deporte|         1550.0|
| Valencia|Electrónica|         1085.0|
| Valencia|      Hogar|          630.0|
+---------+-----------+---------------+


### 🎯 Pregunta 10 — Análisis multidimensional con `cube`

In [62]:
marketNova
  .cube("ciudad", "categoria")
  .agg(sum("total").as("total_facturado"))
  .orderBy($"ciudad", $"categoria")
  .show(50)

+---------+-----------+---------------+
|   ciudad|  categoria|total_facturado|
+---------+-----------+---------------+
|     NULL|       NULL|        21265.0|
|     NULL|    Deporte|         5230.0|
|     NULL|Electrónica|        12195.0|
|     NULL|      Hogar|         3840.0|
|Barcelona|       NULL|         4350.0|
|Barcelona|    Deporte|          570.0|
|Barcelona|Electrónica|         3050.0|
|Barcelona|      Hogar|          730.0|
|   Bilbao|       NULL|         3530.0|
|   Bilbao|    Deporte|          850.0|
|   Bilbao|Electrónica|         2100.0|
|   Bilbao|      Hogar|          580.0|
|   Madrid|       NULL|         6570.0|
|   Madrid|    Deporte|         1250.0|
|   Madrid|Electrónica|         4420.0|
|   Madrid|      Hogar|          900.0|
|  Sevilla|       NULL|         3550.0|
|  Sevilla|    Deporte|         1010.0|
|  Sevilla|Electrónica|         1540.0|
|  Sevilla|      Hogar|         1000.0|
| Valencia|       NULL|         3265.0|
| Valencia|    Deporte|         1550.0|


### 🎯 Pregunta 11 — Tabla dinámica con `pivot`

In [63]:
marketNova
  .groupBy("ciudad")
  .pivot("categoria")
  .agg(sum("total"))
  .orderBy("ciudad")
  .show()

+---------+-------+-----------+------+
|   ciudad|Deporte|Electrónica| Hogar|
+---------+-------+-----------+------+
|Barcelona|  570.0|     3050.0| 730.0|
|   Bilbao|  850.0|     2100.0| 580.0|
|   Madrid| 1250.0|     4420.0| 900.0|
|  Sevilla| 1010.0|     1540.0|1000.0|
| Valencia| 1550.0|     1085.0| 630.0|
+---------+-------+-----------+------+



### 🎯 Pregunta 12 — Guardar resumen por ciudad en CSV

In [64]:
val resumenMNCiudad = marketNova
  .groupBy("ciudad")
  .agg(
    count("id").as("num_ventas"),
    sum("total").as("total_facturado"),
    avg("total").as("ticket_medio")
  )
  .orderBy($"total_facturado".desc)

resumenMNCiudad.write
  .mode("overwrite")
  .option("header", "true")
  .csv("salidas/marketnova_facturacion_ciudad")

println("✅ Guardado en salidas/marketnova_facturacion_ciudad")

✅ Guardado en salidas/marketnova_facturacion_ciudad


resumenMNCiudad: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [ciudad: string, num_ventas: bigint ... 2 more fields]

### 🎯 Pregunta 13 — Pipeline resumen por categoría → CSV

In [65]:
val resumenMNCategoria = marketNova
  .groupBy("categoria")
  .agg(
    count("id").as("num_ventas"),
    sum("cantidad").as("unidades_vendidas"),
    sum("total").as("facturacion_total"),
    avg("total").as("ticket_medio")
  )
  .orderBy($"facturacion_total".desc)

resumenMNCategoria.show()

resumenMNCategoria.write
  .mode("overwrite")
  .option("header", "true")
  .csv("salidas/marketnova_resumen_categoria")

println("✅ Guardado en salidas/marketnova_resumen_categoria")

+-----------+----------+-----------------+-----------------+------------------+
|  categoria|num_ventas|unidades_vendidas|facturacion_total|      ticket_medio|
+-----------+----------+-----------------+-----------------+------------------+
|Electrónica|        13|               35|          12195.0| 938.0769230769231|
|    Deporte|        11|               46|           5230.0|475.45454545454544|
|      Hogar|        12|               36|           3840.0|             320.0|
+-----------+----------+-----------------+-----------------+------------------+

✅ Guardado en salidas/marketnova_resumen_categoria


resumenMNCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string, num_ventas: bigint ... 3 more fields]